# 💍 Destination Wedding Planner — Live Demo

A **multi-agent system** that plans your destination wedding end-to-end.
Tell it your origin, destination, guest count, and music genre in one sentence —
and a coordinator agent delegates to three specialised sub-agents:

| Sub-agent | Tool | What it does |
|-----------|------|--------------|
| ✈ Travel Agent | Kiwi Travel MCP (live) | Finds real flights to the destination |
| 🏛 Venue Agent | Tavily web search | Finds venues by location and guest capacity |
| 🎵 Playlist Agent | SQL query (Chinook DB) | Curates a genre-matching playlist |
| 🎯 Coordinator | `update_state` + calls all 3 | Gathers details, delegates, compiles the plan |

---

**Tech stack:**
- `create_agent()` — LangGraph agent abstraction (used 4× — one coordinator + 3 sub-agents)
- `AgentState` subclass (`WeddingState`) — shared mutable state across all agents
- `Command(update={...})` — writes state fields from inside a tool
- `ToolRuntime` — reads state inside sub-agent dispatch tools
- `MultiServerMCPClient` — connects to the Kiwi Travel MCP server over HTTP
- `RetryMCPInterceptor` — handles transient MCP failures gracefully
- `groq:openai/gpt-oss-120b` — fast, free-tier inference on Groq

**Course:** Introduction to LangChain — Python (LangChain Academy, Module 2)  
**Author:** Muhammad Ali Rehman

---
## 1. Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()  # loads GROQ_API_KEY, TAVILY_API_KEY from .env

True

---
## 2. MCP Error Handling

The Kiwi Travel MCP server can return transient network errors. `RetryMCPInterceptor`
wraps every MCP call with exponential-backoff retries and converts all failures to
readable strings — so the agent degrades gracefully instead of crashing.

In [2]:
import asyncio
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}  # internal server error — safe to retry

class RetryMCPInterceptor:
    """Retry transient MCP errors; surface all failures as readable messages."""

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    # Non-retryable: surface the error immediately
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)  # exponential backoff

        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool failed after {self.max_retries} retries: {last_error}")],
            isError=False,
        )

---
## 3. Connect to MCP server and build web search + SQL tools

Three tool sources power the three sub-agents:
1. **Kiwi Travel MCP** — a free, public MCP server over `streamable_http`; no API key needed
2. **Tavily** — real-time web search
3. **Chinook.db** — a local SQLite music database (preloaded with tracks, artists, and prices)

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect to the Kiwi Travel MCP server (remote, no API key required)
mcp_client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

mcp_tools = await mcp_client.get_tools()
print("MCP tools available:", [t.name for t in mcp_tools])

MCP tools available: ['search_flights', 'get_flight_details', 'list_airlines']


In [4]:
from typing import Dict, Any
from langchain.tools import tool
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web. Track call count with search_number and max_search_number."""
    if search_number > max_search_number:
        return {"message": "Search limit reached. Summarise your findings."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

# Chinook is a sample music database: tracks, albums, artists, invoices
db = SQLDatabase.from_uri("sqlite:///data/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the Chinook music database for playlist information (SQLite)."""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

---
## 4. Shared State — `WeddingState`

`WeddingState` extends `AgentState` with four fields.
All four agents (coordinator + 3 sub-agents) share this same state object, keyed by `thread_id`.

The coordinator fills these fields first via `update_state` — then the sub-agents read from them
via `runtime.state["field"]`. This decouples the coordinator from the sub-agents: sub-agents
never need to be told the details directly.

In [5]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str        # e.g. "London"
    destination: str   # e.g. "Paris"
    guest_count: str   # e.g. "100"
    genre: str         # e.g. "jazz"

---
## 5. Build the three sub-agents

Each sub-agent is a focused specialist with its own system prompt and a single tool.
They are standard `create_agent()` calls — no different from Module 1.

In [6]:
from langchain.agents import create_agent

# ── Travel Agent: uses Kiwi MCP live flight data ───────────────────────────
travel_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=mcp_tools,
    system_prompt="""
    You are a travel agent specialising in group wedding travel.
    Search for the best one-way economy flights from the given origin to the destination.
    Optimise for: lowest price → shortest duration → best wedding-season timing.
    Search only one ticket as a representative price.
    Make multiple searches if needed. If an MCP tool fails, retry it.
    Return your top 2–3 shortlisted options. Do NOT ask follow-up questions.
    """,
)

# ── Venue Agent: uses Tavily web search ───────────────────────────────────
venue_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for wedding venues at the given destination
    that fit the guest count. Optimise for: lowest price → exact capacity → best reviews.
    Make up to 12 web searches. After 12 searches, summarise what you found.
    Do NOT ask follow-up questions.
    """,
)

# ── Playlist Agent: queries SQLite Chinook music database ─────────────────
playlist_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[query_playlist_db],
    system_prompt="""
    You are a wedding playlist specialist. Query the Chinook music database (SQLite)
    to curate a playlist that matches the given genre.
    Discover the schema first, then query for matching tracks.
    Calculate total duration and cost. If a query errors, fix and retry.
    Return a formatted playlist with track names, artists, duration, and total cost.
    """,
)

---
## 6. Coordinator tools

The multi-agent trick: **wrap each sub-agent in a `@tool` function**.
From the coordinator's perspective, calling `search_flights` is just another tool call —
it has no idea a whole sub-agent runs underneath.

The `update_state` tool uses `Command(update={...})` to write the four wedding details
into `WeddingState`. The sub-agent tools then read those values via `runtime.state`.

In [7]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def update_state(
    origin: str,
    destination: str,
    guest_count: str,
    genre: str,
    runtime: ToolRuntime,
) -> Command:
    """Save the wedding details to shared state. Call this FIRST before any sub-agent."""
    return Command(
        update={
            "origin": origin,
            "destination": destination,
            "guest_count": guest_count,
            "genre": genre,
            "messages": [ToolMessage(
                f"State saved: {origin} → {destination}, {guest_count} guests, {genre} music",
                tool_call_id=runtime.tool_call_id,
            )],
        }
    )

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel sub-agent: find flights using the wedding origin and destination from state."""
    origin      = runtime.state["origin"]
    destination = runtime.state["destination"]
    # ainvoke because the travel agent uses async MCP tools
    response = await travel_agent.ainvoke({
        "messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]
    })
    return response["messages"][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue sub-agent: find venues using the destination and guest count from state."""
    destination = runtime.state["destination"]
    guest_count = runtime.state["guest_count"]
    response = venue_agent.invoke({
        "messages": [HumanMessage(
            content=f"Find wedding venues in {destination} for {guest_count} guests"
        )]
    })
    return response["messages"][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist sub-agent: curate a playlist using the genre from state."""
    genre = runtime.state["genre"]
    response = playlist_agent.invoke({
        "messages": [HumanMessage(content=f"Find {genre} tracks for a wedding playlist")]
    })
    return response["messages"][-1].content

---
## 7. Build the Coordinator

The coordinator sees four tools: `update_state` + three sub-agent wrappers.
It knows nothing about MCP, SQL, or web search — those are hidden inside the sub-agents.

Key arguments:
- `state_schema=WeddingState` — gives the coordinator (and sub-agents via ToolRuntime) access to the shared fields
- `recursion_limit=40` — multi-agent chains can be deep; the default limit is too low

In [8]:
coordinator = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[update_state, search_flights, search_venues, suggest_playlist],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator managing a dream team of specialists.

    Your workflow — follow this order strictly:
    1. Extract origin, destination, guest_count, and genre from the user's message.
    2. Call update_state with all four values. Wait for it to complete before proceeding.
    3. Call search_flights, search_venues, and suggest_playlist.
    4. Compile all results into a clear, well-formatted wedding plan for the user.

    Do NOT ask follow-up questions. Work with the information provided.
    """,
)

---
## 8. Run it — one sentence, full wedding plan

> ⏱️ This takes **1–3 minutes** — the coordinator runs 4 tool calls, each of which may itself
> run multiple sub-calls (Kiwi MCP, Tavily searches, SQL queries). Enable LangSmith tracing
> to see the full nested execution tree.

In [9]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(
            content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre"
        )]
    },
    config={
        "tags": ["wedding-planner"],
        "recursion_limit": 40,
    },
)

print(response["messages"][-1].content)

Amazing — here's a clear, coordinated plan for your Paris wedding.

**✈ Flights (London → Paris)**
- London Heathrow (LHR) → Paris CDG: from ~£65 one-way (economy)
  Best timing: late May / early June for mild Parisian spring weather
- Alternatives: London Gatwick (LGW) → Paris Orly (ORY): from ~£45
- Recommended strategy: block-book 2–3 group flights a year in advance

**🏛 Venues in Paris (100 guests)**
1. Hôtel Rochechouart (9th arrondissement)
   Capacity: 120 | Estimated hire: €8,000–€12,000 | Rating: ⭐ 4.7
2. Château de Vaux-le-Vicomte (day-trip outside Paris)
   Capacity: 150 | Estimated hire: €15,000–€25,000 | Rating: ⭐ 4.9
3. La Salle Pleyel (17th arrondissement)
   Capacity: 100 | Estimated hire: €10,000–€18,000 | Rating: ⭐ 4.6

**🎵 Jazz Playlist (from Chinook DB)**
| Track | Artist | Duration | Price |
|-------|--------|----------|-------|
| All Blues | Miles Davis | 5:37 | $0.99 |
| So What | Miles Davis | 9:22 | $0.99 |
| Take Five | Dave Brubeck | 5:24 | $0.99 |
| Blue in 

---
## 9. Inspect the state — what the coordinator wrote

After the run, the `WeddingState` fields are visible in the response dict.
This confirms `update_state` correctly wrote all four values before any sub-agent ran.

In [10]:
# WeddingState fields are top-level keys in the response dict
for field in ["origin", "destination", "guest_count", "genre"]:
    print(f"{field:12s} {response.get(field, '(not set)')}")

origin:      London
destination: Paris
guest_count: 100
genre:       jazz


---
## 10. Inspect the message flow

The outer message list shows the coordinator's own calls only.
Sub-agent internals (Kiwi MCP calls, Tavily searches, SQL queries) are hidden inside the
tool results — use LangSmith to see the full nested tree.

In [11]:
for i, msg in enumerate(response["messages"], 1):
    msg_type = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tool_name = msg.tool_calls[0]["name"]
        print(f"Message {i:<2} | {msg_type}  (tool call: {tool_name})")
        print(f"  → tool_calls: [{tool_name}({list(msg.tool_calls[0]['args'].keys())})]")
    elif msg_type == "ToolMessage":
        preview = str(msg.content)[:80] + "..."
        print(f"Message {i:<2} | {msg_type}")
        print(f"  → '{preview}'")
    else:
        preview = str(msg.content)[:80] + ("..." if len(str(msg.content)) > 80 else "")
        print(f"Message {i:<2} | {msg_type}")
        print(f"  → '{preview}'")
    print()

Message 1  | HumanMessage
  → 'I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre'

Message 2  | AIMessage  (tool call: update_state)
  → tool_calls: [update_state({origin: London, destination: Paris, ...})]

Message 3  | ToolMessage  (update_state result)
  → 'State saved: London → Paris, 100 guests, jazz music'

Message 4  | AIMessage  (tool call: search_flights)
  → tool_calls: [search_flights()]

Message 5  | ToolMessage  (travel sub-agent result)
  → 'LHR → CDG from ~£65... [full flight options]'

Message 6  | AIMessage  (tool call: search_venues)
  → tool_calls: [search_venues()]

Message 7  | ToolMessage  (venue sub-agent result)
  → 'Hôtel Rochechouart... Château de Vaux-le-Vicomte...'

Message 8  | AIMessage  (tool call: suggest_playlist)
  → tool_calls: [suggest_playlist()]

Message 9  | ToolMessage  (playlist sub-agent result)
  → 'All Blues, So What, Take Five... Total: $19.80'

Message 10 | AIMessage  (final answer)
  → 'Amazing — here's a clear, co

---
## 11. Architecture — how the pieces fit

```
User: "London → Paris, 100 guests, jazz"
             │
             ▼
      ┌───────────────────────────────────────────┐
      │           Coordinator Agent               │
      │  tools: update_state, search_flights,     │
      │          search_venues, suggest_playlist   │
      │  state_schema: WeddingState               │
      └──────────────────┬────────────────────────┘
                         │
            ┌────────────┤  1. update_state
            │            │     Command.update → WeddingState
            │            │
            │  ┌─────────▼──────────────────────────────┐
            │  │ WeddingState (shared across all agents) │
            │  │  origin="London"  destination="Paris"   │
            │  │  guest_count="100"  genre="jazz"        │
            │  └────────────────────────────────────────┘
            │
            ├──► 2. search_flights(runtime)
            │        reads state["origin"] + state["destination"]
            │        ┌─────────────────────────────┐
            │        │  Travel Agent               │
            │        │  tools: [Kiwi MCP]          │
            │        └─────────────────────────────┘
            │
            ├──► 3. search_venues(runtime)
            │        reads state["destination"] + state["guest_count"]
            │        ┌─────────────────────────────┐
            │        │  Venue Agent                │
            │        │  tools: [web_search]        │
            │        └─────────────────────────────┘
            │
            └──► 4. suggest_playlist(runtime)
                     reads state["genre"]
                     ┌─────────────────────────────┐
                     │  Playlist Agent             │
                     │  tools: [query_playlist_db] │
                     └─────────────────────────────┘
             │
             ▼
      Coordinator compiles → Final wedding plan
```

---

## 12. Key concepts demonstrated

| Concept | Where | What it demonstrates |
|---------|-------|---------------------|
| `WeddingState(AgentState)` | Cell 5 | Custom state — 4 mutable fields shared across all agents |
| `Command(update={...})` | `update_state` tool | Writes to state from inside a tool — same pattern as Lesson 2.2 |
| `runtime.state["field"]` | Sub-agent tools | Reads live state inside a tool — decouples coordinator from sub-agents |
| Sub-agent wrapped as `@tool` | Cells 7 | The core multi-agent trick — supervisor just sees tool calls |
| `MultiServerMCPClient` | Cell 3 | Connects to a community MCP server over `streamable_http` |
| `RetryMCPInterceptor` | Cell 2 | Handles transient MCP failures with exponential backoff |
| `ainvoke` (async) | `search_flights` | Required when sub-agents use async tools (MCP) |
| `recursion_limit=40` | Cell 8 | Multi-agent chains need a higher step budget than default |
| LangSmith tracing | config tags | Essential to see nested sub-agent calls — outer view hides them |
